<a href="https://colab.research.google.com/github/karthiksagarN/Bart-FineTuned-Model-406M/blob/main/finetuning_bart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset, DatasetDict #, load_from_disk
from transformers import TrainingArguments, Trainer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import uuid
import torch


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

In [ ]:
dataset = load_dataset("gretelai/gretel-financial-risk-analysis-v1")

# 10% of train → validation
train_valid = dataset["train"].train_test_split(test_size=0.10, seed=42)

dataset = DatasetDict({
    "train": train_valid["train"],
    "validation": train_valid["test"],
    "test": dataset["test"],
})

#print(dataset)

In [ ]:
def simplify_record(example):
    return {
        "id": str(uuid.uuid4())[:8],
        "dialogue": example["input"],
        "summary": (
            example["output"]["analysis"]
            if isinstance(example["output"], dict) and "analysis" in example["output"]
            else str(example["output"])
        ),
    }

# Apply to each split in the DatasetDict
simplified_splits = {}
for split_name, split_data in dataset.items():
    simplified_splits[split_name] = split_data.map(
        simplify_record,
        remove_columns=split_data.column_names
    )

dataset = DatasetDict(simplified_splits)

print(dataset)
#print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 744
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 83
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 207
    })
})


In [ ]:
sample = dataset['test'][3]['dialogue']
label = dataset['test'][3]['summary']

def generate_summary(input , llm):
    input_prompt = f"""
                    Summarize the following conversation.
                    {input}
                    Summary:
                    """
    input_ids = tokenizer(sample, return_tensors='pt')
    tokenized_output = llm.generate(input_ids['input_ids'], min_length=30, max_length=200)
    output = tokenizer.decode(tokenized_output[0], skip_special_tokens=True)

    return output

In [ ]:
#summerize an example with the facebook model
output = generate_summary(sample, llm=model)
print("Sample")
print(sample)
print("--------------------")
print("Model Generated Summary:")
print(output)
print("Correct Summary:")
print(label)

In [ ]:
def tokenize_inputs(example):
    start_prompt = "Summarize the following conversation. \n\n"
    end_prompt = "\n\nSummary: "
    prompt = [start_prompt + dialogue + end_prompt for dialogue in example['dialogue']]
    
    # Set max_length for BART 
    max_length = 512
    
    example['input_ids'] = tokenizer(prompt, padding='max_length', truncation=True, max_length=max_length, return_tensors='pt').input_ids
    example['labels'] = tokenizer(example['summary'], padding="max_length", truncation=True, max_length=max_length, return_tensors='pt').input_ids

    return example

# Set pad_token before tokenization
tokenizer.pad_token = tokenizer.eos_token
tokenized_datasets = dataset.map(tokenize_inputs, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(['id', 'dialogue', 'summary'])
tokenized_datasets = tokenized_datasets.filter(lambda example, index : index%100==0, with_indices=True)

Map:   0%|          | 0/744 [00:00<?, ? examples/s]

Map:   0%|          | 0/83 [00:00<?, ? examples/s]

Map:   0%|          | 0/207 [00:00<?, ? examples/s]

Filter:   0%|          | 0/744 [00:00<?, ? examples/s]

Filter:   0%|          | 0/83 [00:00<?, ? examples/s]

Filter:   0%|          | 0/207 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_datasets['train'].shape)
print(tokenized_datasets['test'].shape)
print(tokenized_datasets['validation'].shape)

In [ ]:
training_args = TrainingArguments(
    output_dir="./bart-finetuned",
    hub_model_id="karthiksagarn/bart-finetuned",
    learning_rate=3e-5,
    num_train_epochs=20,
    weight_decay=0.01,
    auto_find_batch_size=True,
    # per_device_train_batch_size=8,  # Start with a smaller batch size
    warmup_steps=100,
    warmup_ratio=0.1, #
    evaluation_strategy='epoch',
    logging_strategy="epoch",
    logging_steps=10

)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation']
)

In [ ]:
trainer.train()

C:\Users\pcyye\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,5.950200,4.685424
2,5.915000,4.466804
3,5.544800,4.069008
4,5.182200,3.540890
5,4.374100,2.951970
6,3.492200,2.375476
7,2.733300,1.810710
8,2.062200,1.265299
9,1.348600,0.802395
10,0.844400,0.484313


C:\Users\pcyye\AppData\Roaming\Python\Python313\site-packages\transformers\modeling_utils.py:2810: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
C:\Users\pcyye\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=20, training_loss=1.967534303292632, metrics={'train_runtime': 1788.8594, 'train_samples_per_second': 0.089, 'train_steps_per_second': 0.011, 'total_flos': 173368368168960.0, 'train_loss': 1.967534303292632, 'epoch': 20.0})

In [ ]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.1100718230009079, 'eval_runtime': 1.7787, 'eval_samples_per_second': 0.562, 'eval_steps_per_second': 0.562, 'epoch': 20.0}


In [ ]:
output = generate_summary(sample, llm=model)
print("Sample")
print(sample)
print("--------------------")
print("Model Generated Summary:")
print(output)
print("--------------------")
print("Dataset Summary:")
print(label)

Sample
"of $5.0 billion in 2022, $5.2 billion in 2021, and $5.1 billion in 2020. As of December 31, 2022, $4.3 billion of the $5.0 billion of 2022 notes remained outstanding. The notes that were repaid in 2022 were settled primarily through cash payments, with a minor portion being exchanged for other debt instruments with more favorable terms.

We also issue commercial paper, which is generally used to finance our short-term working capital requirements. We have issued commercial paper of up to $10.0 billion in aggregate principal amount. As of December 31, 2022, $5.6 billion of commercial paper was outstanding. This commercial paper is primarily used to fund our operational needs, such as inventory and accounts receivable, and we generally repay these borrowings within a short period, typically within 90 days. The interest rates on our commercial paper are determined by market conditions and our credit rating, and we closely monitor our commercial paper borrowings to minimize the cos

In [ ]:
# Create a local folder path
save_path = "./bart_summarization_model_v3"

# Save model and tokenizer locally
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved to:", save_path)

Model saved to: ./bart_summarization_model_v3


### Testing the Model

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained("./bart_summarization_model_v3")
tokenizer = AutoTokenizer.from_pretrained("./bart_summarization_model_v3")

# 2) Put model in eval mode and on the right device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def summarize(text, max_input_len=512, max_new_tokens=128):
    # 3) Proper tokenization with attention_mask, on the same device
    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_input_len,
        return_tensors="pt"
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    # 4) Generate with sensible defaults
    with torch.no_grad():
        out_ids = model.generate(
            **enc,
            num_beams=4,
            max_new_tokens=max_new_tokens,
            length_penalty=2.0,
            early_stopping=True
        )
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)

Sample
 Our ability to generate sufficient cash to meet our obligations, including the payment of interest on our outstanding debt, may be impaired. If we are unable to generate sufficient cash to meet our obligations, we may be required to seek additional capital, restructure our debt, or reduce our operations, which could negatively impact our business, results of operations, and financial condition.

We may not be able to generate sufficient cash to meet our obligations, including the payment of interest on our outstanding debt, which could negatively impact our business, results of operations, and financial condition. Our cash flows from operations are subject to various factors, including the demand for our products and services, competition in the market, the level of our operating expenses, and the availability of raw materials and supplies. Any significant decline in our cash flows from operations could have a material adverse effect on our ability to meet our obligations and m

In [27]:
# --- sample usage ---
sample = dataset["test"][15]["dialogue"]
label  = dataset["test"][15]["summary"]

pred = summarize(sample)
print("Sample\n", sample)
print("\nModel Generated Summary:\n", pred)
print("\nDataset Summary:\n", label)

Sample
 Our ability to generate sufficient cash to meet our obligations, including the payment of interest on our outstanding debt, may be impaired. If we are unable to generate sufficient cash to meet our obligations, we may be required to seek additional capital, restructure our debt, or reduce our operations, which could negatively impact our business, results of operations, and financial condition.

We may not be able to generate sufficient cash to meet our obligations, including the payment of interest on our outstanding debt, which could negatively impact our business, results of operations, and financial condition. Our cash flows from operations are subject to various factors, including the demand for our products and services, competition in the market, the level of our operating expenses, and the availability of raw materials and supplies. Any significant decline in our cash flows from operations could have a material adverse effect on our ability to meet our obligations and m

### Evaluation Metrics

In [ ]:
import evaluate #from datasets import load_metric
from tqdm import tqdm, trange
import pandas
from bert_score import score
device = "cpu"

model = AutoModelForSeq2SeqLM.from_pretrained("./bart_summarization_model_v3")
tokenizer = AutoTokenizer.from_pretrained("./bart_summarization_model_v3")

C:\Users\pcyye\AppData\Roaming\Python\Python313\site-packages\transformers\models\bart\configuration_bart.py:176: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(


In [ ]:
rouge_metric = evaluate.load("rouge")

def summarize(text: str, model, tokenizer, *,
              max_input_len: int = 512,
              max_new_tokens: int = 128,
              num_beams: int = 4) -> str:
    """Generate a summary for one text with sensible defaults."""
    model.eval()
    device = next(model.parameters()).device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_input_len,
        return_tensors="pt"
    ) #
    enc = {k: v.to(device) for k, v in enc.items()} #

    with torch.no_grad():
        out_ids = model.generate(
            **enc,
            num_beams=num_beams,
            max_new_tokens=max_new_tokens,
            length_penalty=2.0,
            early_stopping=True,
            no_repeat_ngram_size=3
        ) #
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)


def compute_rouge_scores(dataset_split, model, tokenizer):
    """Compute ROUGE on a HF Dataset split with 'dialogue' and 'summary' fields."""
    predictions, references = [], []

    for example in dataset_split:
        dialogue = example["dialogue"]
        reference = example["summary"]

        pred = summarize(dialogue, model=model, tokenizer=tokenizer)

        predictions.append(pred)
        references.append(reference)

    return rouge_metric.compute(predictions=predictions, references=references)


# Compute ROUGE scores for validation dataset
rouge_scores = compute_rouge_scores(dataset['test'], model, tokenizer)

print("ROUGE Scores:")
print(rouge_scores)

C:\Users\pcyye\AppData\Roaming\Python\Python313\site-packages\transformers\generation\utils.py:1532: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


ROUGE Scores:
{'rouge1': np.float64(0.1833416709657167), 'rouge2': np.float64(0.05744062317065032), 'rougeL': np.float64(0.1494559428446432), 'rougeLsum': np.float64(0.14909507337743558)}


In [13]:
tokenizer1 = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model1 = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

In [ ]:
rouge_scores = compute_rouge_scores(dataset['validation'], model1, tokenizer1)

print("ROUGE Scores:")
print(rouge_scores)

ROUGE Scores:
{'rouge1': np.float64(0.19660883505037458), 'rouge2': np.float64(0.08006262120458782), 'rougeL': np.float64(0.15938826242563886), 'rougeLsum': np.float64(0.1591981239245565)}


In [10]:
def compute_bert_score(dataset_split, model, tokenizer, *,
                       max_input_len=512, max_new_tokens=128,
                       num_beams=4, bert_model='bert-base-uncased'):
    """
    Compute BERTScore for a dataset split using a summarization model.

    Args:
        dataset_split: HF Dataset split with 'dialogue' and 'summary' fields
        model: Seq2Seq model (e.g., BART/T5)
        tokenizer: corresponding tokenizer
        max_input_len: max input length for summarization
        max_new_tokens: number of tokens to generate
        num_beams: beam width for generation
        bert_model: which BERT model to use for BERTScore
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()

    predictions, references = [], []

    print(f"Generating summaries for {len(dataset_split)} samples...")

    for example in dataset_split:
        text = example["dialogue"]
        ref_summary = example["summary"]

        # Tokenize & generate summary
        inputs = tokenizer(
            text,
            truncation=True,
            max_length=max_input_len,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                num_beams=num_beams,
                max_new_tokens=max_new_tokens,
                length_penalty=2.0,
                early_stopping=True,
                no_repeat_ngram_size=3
            )

        gen_summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        predictions.append(gen_summary)
        references.append(ref_summary)

    print("Computing BERTScore...")
    P, R, F1 = score(predictions, references, lang="en", model_type=bert_model)
    print(f"\n✅ Evaluated {len(predictions)} examples.\n")

    return {
        "precision": P.mean().item(),
        "recall": R.mean().item(),
        "f1": F1.mean().item()
    }

In [11]:
# === Example usage ===
bert_scores = compute_bert_score(dataset["test"], model, tokenizer)
print(f"BERTScore - Precision: {bert_scores['precision']:.4f}, "
      f"Recall: {bert_scores['recall']:.4f}, "
      f"F1: {bert_scores['f1']:.4f}")

Generating summaries for 207 samples...
Computing BERTScore...


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

C:\Users\pcyye\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pcyye\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)



✅ Evaluated 207 examples.

BERTScore - Precision: 0.4846, Recall: 0.5118, F1: 0.4956


In [14]:
# === Example usage ===
bert_scores = compute_bert_score(dataset["test"], model1, tokenizer1)
print(f"BERTScore - Precision: {bert_scores['precision']:.4f}, "
      f"Recall: {bert_scores['recall']:.4f}, "
      f"F1: {bert_scores['f1']:.4f}")

Generating summaries for 207 samples...
Computing BERTScore...

✅ Evaluated 207 examples.

BERTScore - Precision: 0.4638, Recall: 0.5854, F1: 0.5161
